# SHAP — Depth Sweep (Notebook 3 / 3)

Runs the **SHAP package** (`shap.TreeExplainer`) across all datasets and task types
in the `woodelfhd_depth_sweep_experiment`.  
This notebook is **independent** and can run in parallel with Notebooks 01 and 02.
Notebook 04 merges all three result files and generates the HTML report.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` and `woodelf_explainer` repos
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method shap`
5. Writes partial results to Drive as `partial_shap.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | UCI direct download (~8 GB uncompressed — takes ~20 min) |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

### SHAPApproach behaviour in this experiment
- `background_shap_interactions`: not supported by the shap library — recorded as `not_supported`.
- `background_shap`: uses `bg_shap_limit=10` background rows (configured in the experiment);
  times are extrapolated when the full background set is larger.
- `woodelf_explainer` is still needed (it is a dependency of `treebranchmarks`).

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
# Must match the DRIVE_FOLDER used in notebooks 01, 02, and 04.
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'partial_shap.json'
print(f'Results will be saved to: {DRIVE_RESULT_PATH}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 5: Run the experiment (SHAP only) ───────────────────────────────────
# --method shap : only the SHAPApproach is timed
# The method name 'shap' matches SHAP.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method shap \
    --result_location "{DRIVE_RESULT_PATH}"

In [ ]:
# ── Step 6: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    data = json.load(f)

n_missions    = len(data['missions'])
n_task_results = sum(len(m['task_results']) for m in data['missions'])

approach_names = set()
for m in data['missions']:
    for tr in m['task_results']:
        approach_names.update(tr['approach_results'].keys())

print(f'Missions:      {n_missions}')
print(f'Task results:  {n_task_results}')
print(f'Approaches:    {sorted(approach_names)}')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')